In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)

True

In [3]:
import os

print(os.getcwd())

/mnt/data/llm_project/agentic_workflow_lab/notebooks


In [4]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [5]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"


gemini = OpenAI(base_url=gemini_url, api_key=google_api_key)
ollama = OpenAI(base_url=ollama_url, api_key="ollama",)
groq = OpenAI(base_url=groq_url, api_key=groq_api_key)

In [6]:
reader = PdfReader("../me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text


print(linkedin)

   
Contact
deepakdeepu4477@gmail.com
www.linkedin.com/in/deepak-
lingaraju-a4702b275 (LinkedIn)
Top Skills
Signal Processing
XAI
Feature Engineering
Languages
English (Full Professional)
Kannada (Native or Bilingual)
German (Professional Working)
Certifications
Python for Data Science and
Machine Learning Bootcamp
Optimization Onramp
Introduction to Statistical Methods
with MATLAB
Reinforcement Learning Onramp
Machine Learning with MATLAB
Deepak Lingaraju
Computer Vision enthusiast pursuing Mechatronics Engineering
Masters Degree | Deep Learning | AI | Python.
Germany
Summary
I’m passionate about developing AI-driven systems for automation,
robotics, and intelligent manufacturing, integrating machine learning,
deep learning, and computer vision with real-world engineering
challenges.
My interests include robot perception, sensor fusion, and
autonomous decision-making, with hands-on experience working in
ROS 2 environments that bridge deep learning and computer vision
for intelligent r

In [7]:
with open("../me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [8]:
name = "Deepak Lingaraju"

In [9]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.2:latest", messages=messages)
    return response.choices[0].message.content

In [11]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [12]:
from pydantic import BaseModel
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [13]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [14]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [15]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [16]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = ollama.chat.completions.create(model="llama3.2:latest", messages=messages)
reply = response.choices[0].message.content

In [17]:
reply

'Patents! Yes, I do hold a patent. As part of my Master\'s studies in Mechatronics Engineering, I worked on a project that led to the development of a novel algorithm for object detection and tracking in industrial settings. After conducting rigorous research and experimentation, I successfully filed a patent application for my invention.\n\nThe patent, titled "Robust Object Detection and Tracking for Industrial Automation," focuses on improving the accuracy and efficiency of computer vision-based systems used in industrial environments. My algorithm uses a combination of deep learning and computer vision techniques to detect and track objects with high precision, even in complex and dynamic environments.\n\nI\'m really proud of this achievement, as it demonstrates the potential of AI and computer vision to improve industrial automation and increase productivity. Who knows? Maybe one day, my patented technology will make a significant impact in the manufacturing industry!'

In [18]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=False, feedback='The agent hallucinates information about holding a patent. The provided context (summary and LinkedIn profile) does not mention any patents held by Deepak Lingaraju. The agent should have stated that it does not know the answer or that no such information is available in the provided context.')

In [19]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.2:latest", messages=messages)
    return response.choices[0].message.content

In [24]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.2:latest", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
The agent's response is entirely in Pig Latin, which is completely unprofessional and inappropriate for the persona of Deepak Lingaraju, who is representing himself to potential clients or employers. This does not align with the instruction to be professional and engaging. The agent should respond in clear, professional English.
